### The purpose of this script is to evaluate the test dataset with an LLM model.

In [13]:
#importing packages
import json
import pickle
import pandas as pd
import openai
from openai import OpenAI
import numpy as np
import tabulate

#### Creating testResult object class

In [3]:
class testResult:
    def __init__(self, model_name_short, dataset, prompt, promptType):
        self.model_name_short = model_name_short
        self.dataset = dataset
        self.prompt = prompt
        self.promptType = promptType
        self.time = None
        self.result = None
        self.reranking_score_1 = None
        self.reranking_score_2 = None
        self.retrival_score = None

    def set_result_df(self, result):
        self.result = result

    def set_time(self, time):
        self.time = time  

    def set_reranking_1(self, reranking_score_1):
        self.reranking_score_1 = reranking_score_1

    def set_reranking_1(self, reranking_score_2):
        self.reranking_score_2 = reranking_score_2

    def set_retrieval(self, retrieval_score):
        self.retrieval_score = retrieval_score

In [ ]:
#loading in testing data
input_filename = r'.\LLM as a Judge Files\test_results'

dat = open(input_filename, 'rb') 
loaded_data = pickle.load(dat)
dat.close()

print(len(loaded_data))

840


In [ ]:
openai_key = my_key

#### Creating function and testing model with prompt

#### Creating prompt, processing function

In [ ]:
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.1

SYSTEM_PROMPT = """
You are tasked with evaluating the relevance of a conference program abstract (Result) to a specific search query (Query). Your evaluation must consider two distinct criteria: Application Relevance and Methodology Relevance.

Input:
You will be given a Query and a Result.

The Query can be in various formats:
- A set of keywords (e.g., "supply chain optimization")
- A question (e.g., "How can simulation improve hospital workflows?")
- A full conference abstract (where the goal is to find similar abstracts)

The Result will be a conference abstract.

Output:
Return a JSON object in the following format:
{
"Application": <application_score>,
"Methodology": <methodology_score>
}

Scoring Criteria:
Use the following scale (1-3) to assign scores based on the comparison between the criteria identified in the Query and the Result. The <criterion> refers to either 'Application' or 'Methodology'.

3: Highly Relevant
- The Result addresses the exact same specific domain or concept identified for the <criterion> in the Query.
- Synonyms or very closely related terms (e.g., "discrete event modeling" vs. "discrete event simulation") count as highly relevant.
- For Application, closely related fields within a broader category (e.g., 'flood risk' and 'wildfire response' are both 'natural disaster management') can also score 3 if the core challenge is analogous between the Query and Result.

2: Somewhat Relevant
- The Result addresses a related domain or concept for the <criterion>, but not the specific one identified in the Query. This could include:
    - A broader category (e.g., Query criterion is a specific algorithm, Result criterion is the general field like 'Optimization').
    - A sibling field or concept (e.g., Query criterion is 'simulation', Result criterion is 'statistical modeling').
    - A conceptually adjacent field.

1: Irrelevant
- The Result does not contain information relevant to the domain or concept identified for the <criterion> in the Query.

Handling Missing Criteria:
- Based on your analysis of the Query (regardless of its format), determine if it has a primary focus on Application and/or Methodology.
- If the Query does not yield a clear Application focus, set Application score to `Null`.
- If the Query does not yield a clear Methodology focus, set Methodology score to `Null`.

Evaluation Process:
1. Analyze Query:
    - Determine Query Format: Note if it's keywords, a question, or an abstract.
    - Identify Query Criteria: Extract the core Application domain and/or Methodology from the Query.
        - Keywords: Identify terms representing Application/Methodology directly.
        - Question: Determine the main subject (Application) and any mentioned techniques/approaches (Methodology).
        - Abstract: Identify the primary Application domain and the main Methodology described or utilized within the query abstract.
    - Note if either criterion seems absent or not the focus of the query.
2. Analyze Result: Identify the Application domain and Methodology presented in the Result abstract.
3. Compare Application: If an Application criterion was identified in the Query, compare it to the Result's Application using the 1-3 scale. Assign `Null` if the Query lacked a clear Application focus.
4. Compare Methodology: If a Methodology criterion was identified in the Query, compare it to the Result's Methodology using the 1-3 scale. Assign `Null` if the Query lacked a clear Methodology focus.
5. Format Output: Construct the JSON response.

Examples:

Example 1:
Query: Discrete event modeling in flood risk management
Result: Effective wildfire response demands understanding... This presentation introduces Discrete Event Simulation (DES)...
Output:
{
"Application": 3,
"Methodology": 3
}
(Explanation: Query has both App & Method. Result App (wildfire) is related to Query App (flood) -> 3. Result Method (DES) matches Query Method (DEM) -> 3.)

Example 2:
Query: Discrete event modeling in flood risk management
Result: Modern manufacturing environments... Operations Research (OR) provides... optimization... simulation... stochastic modeling...
Output:
{
"Application": 1,
"Methodology": 2
}
(Explanation: Query has both App & Method. Result App (manufacturing) unrelated to Query App (flood) -> 1. Result Method (OR/simulation) is somewhat related to Query Method (DEM) -> 2.)

Example 3:
Query: Discrete event modeling
Result: Educational institutions... effective change management... frameworks like Kotter's 8-Step Process and ADKAR...
Output:
{
"Application": None,
"Methodology": 1
}
(Explanation: Query has Method but NO Application. Result Method (Change Mgmt) is unrelated to Query Method (DEM) -> 1. Since Query lacks Application, Application score MUST be `null`.)

Example 4: 
Query: Supply chain logistics, 
Result: ...wildfire response... Discrete Event Simulation (DES)...)
Output:
{"Application": 1, 
"Methodology": null
}
(Explanation: Query specifies Application ("supply chain logistics") but no Methodology. Result Application ("wildfire response") is irrelevant to the Query Application (Score 1). Since the Query explicitly lacks a Methodology, the Methodology score MUST be `None`, regardless of the DES methodology mentioned in the Result.
"""

def evaluate_relevance(query_text: str, result_abstract: str) -> dict | None:
    """
    Uses OpenAI API to evaluate the relevance of a result abstract to a query.

    Args:
        query_text: The search query (keywords, question, or abstract).
        result_abstract: The conference abstract to evaluate.

    Returns:
        A dictionary with 'Application' and 'Methodology' scores,
        or None if an error occurs.
""" 
    try:
        client = OpenAI(api_key=openai_key) # Initializes client, automatically uses OPENAI_API_KEY env var

        user_message = f"""
        Query:
        {query_text}

        Result Abstract:
        {result_abstract}

        Please evaluate the relevance according to the instructions and provide ONLY the JSON output.
        """

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            temperature=TEMPERATURE,
            max_tokens=150 # Should be enough for the JSON output
        )

        content = response.choices[0].message.content

        # Attempt to parse the response as JSON
        try:
            # Strip potential leading/trailing whitespace or markdown fences
            clean_content = content.strip().strip('```json').strip('```').strip()
            scores = json.loads(clean_content)
            # Basic validation
            if isinstance(scores, dict) and "Application" in scores and "Methodology" in scores:
                # Ensure None values are represented correctly if needed
                if scores["Application"] == "None": scores["Application"] = None
                if scores["Methodology"] == "None": scores["Methodology"] = None
                return scores
            else:
                print(f"Error: Unexpected JSON structure received: {content}")
                return None
        except json.JSONDecodeError:
            print(f"Error: Failed to decode JSON response: {content}")
            return None
        except Exception as e:
            print(f"Error processing response content: {e}")
            print(f"Content received: {content}")
            return None

    except Exception as e:
        print(f"An error occurred during the API call: {e}")
        return None

#### Testing prompt

In [39]:
query1 = "Discrete event modeling in flood risk management"
result1 = """
Effective wildfire response demands understanding the complex interplay between fire behavior, environmental conditions, suppression efforts, and community evacuation dynamics. This presentation introduces Discrete Event Simulation (DES) as a powerful methodology for analyzing these rapidly evolving systems. By representing key occurrences such as fire ignition and spread, changes in weather patterns, dispatch of firefighting resources, and the issuance of evacuation orders as distinct events, DES captures the sequential dependencies critical to wildfire scenarios. Simulating these event chains allows the model to capture complex interactions and the inherent stochastic variability in factors like fire spread and resource effectiveness. This approach enables the quantitative evaluation of different suppression tactics, resource pre-positioning strategies, and evacuation plans under diverse potential fire conditions, ultimately providing incident commanders and planners with a valuable tool to enhance operational planning and improve community safety.
"""
print(f"Evaluating Query 1:\n  Query: '{query1}'\n  Result: (Wildfire DES abstract)")
scores1 = evaluate_relevance(query1, result1)
print(f"Scores: {scores1}\n") # Expected: {'Application': 3, 'Methodology': 3}

# Example 2: Question Query
query2 = "How can simulation improve hospital workflows?"
result2 = """
Modern manufacturing environments are characterized by intense competition, complex supply chains, and the constant need for efficiency improvements. Operations Research (OR) provides a powerful analytical framework to tackle these challenges, enabling data-driven decision-making across various production functions. Techniques such as optimization for production planning and scheduling, simulation for evaluating system performance and bottleneck analysis, and stochastic modeling for inventory control are instrumental. Applying these OR methodologies helps manufacturers minimize costs, optimize resource utilization, improve throughput, and enhance overall operational effectiveness. This presentation will highlight key OR applications and their tangible impact on contemporary manufacturing systems.
"""
print(f"Evaluating Query 2:\n  Query: '{query2}'\n  Result: (Manufacturing OR abstract)")
scores2 = evaluate_relevance(query2, result2)
print(f"Scores: {scores2}\n") # Expected: {'Application': 1, 'Methodology': 2} (App: Hospital vs Manu -> 1, Method: Sim vs OR/Sim -> 2)

# Example 3: Abstract Query
query3 = """
Agent-Based Modeling (ABM) offers a bottom-up approach to understanding complex systems by simulating the actions and interactions of autonomous agents. This presentation explores the application of ABM to model pedestrian dynamics in large public venues during emergency evacuations. We focus on how individual agent behaviors, such as following exit signs, avoiding congestion, and potential panic responses, collectively influence overall evacuation efficiency and safety outcomes. The model allows testing various architectural layouts and guidance strategies.
"""
result3 = """
Educational institutions constantly face changes from technology, policy, and pedagogy, necessitating effective change management for successful adaptation. Navigating these transitions requires addressing unique challenges and resistance inherent in educational settings. This presentation explores frameworks like Kotter's 8-Step Process and ADKAR, highlighting techniques such as strategic communication, stakeholder engagement, and capacity building. Utilizing these strategies empowers educational leaders to manage change effectively, fostering buy-in and ensuring sustainable improvements.
"""
print(f"Evaluating Query 3:\n  Query: (Pedestrian ABM abstract)\n  Result: (Education Change Management abstract)")
scores3 = evaluate_relevance(query3, result3)
print(f"Scores: {scores3}\n") # Expected: {'Application': 1, 'Methodology': 1} (App: Evacuation vs Education -> 1, Method: ABM vs Change Mgmt -> 1)

# Example 4: Query with missing criteria
query4 = "supply chain logistics"
result4 = result1 # Using the wildfire abstract again
print(f"Evaluating Query 4:\n  Query: '{query4}'\n  Result: (Wildfire DES abstract)")
scores4 = evaluate_relevance(query4, result4)
print(f"Scores: {scores4}\n") # Expected: {'Application': 1, 'Methodology': None} (App: Supply Chain vs Wildfire -> 1, Method: Query has none -> None)

Evaluating Query 1:
  Query: 'Discrete event modeling in flood risk management'
  Result: (Wildfire DES abstract)
Scores: {'Application': 3, 'Methodology': 3}

Evaluating Query 2:
  Query: 'How can simulation improve hospital workflows?'
  Result: (Manufacturing OR abstract)
Scores: {'Application': 1, 'Methodology': 2}

Evaluating Query 3:
  Query: (Pedestrian ABM abstract)
  Result: (Education Change Management abstract)
Scores: {'Application': 1, 'Methodology': 1}

Evaluating Query 4:
  Query: 'supply chain logistics'
  Result: (Wildfire DES abstract)
Scores: {'Application': 1, 'Methodology': None}



#### Creating dataframe for batch processing

In [6]:
df = pd.DataFrame()

for res in loaded_data:
    temp_df = res.result[['text']]
    temp_df['prompt'] = res.prompt
    temp_df['promptType'] = res.promptType
    temp_df['time'] = res.time
    temp_df['model_name'] = res.model_name_short
    temp_df['dataset'] = res.dataset

    df = pd.concat([df, temp_df])
df.reset_index(inplace=True, drop=True)
display(df)


,text,prompt,promptType,time,model_name,dataset
0,Quality Assurance Through Layered Process Audi...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024
1,Enhancing supply chain resilience: evaluating ...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024
2,How to succeed large organizational OPEX trans...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024
3,Explore Your Career Choices. This presentatio...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024
4,Industrial Engineers to the Rescue! Solving Yo...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024
...,...,...,...,...,...,...
8395,Data-Driven Performance Guarantees for Classic...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024
8396,Distributionally Robust Optimization Through t...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024
8397,Optimization with Highly Adversarial Gradient ...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024
8398,High probability bounds for stochastic non-con...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024


In [ ]:
df.to_csv(r'.\LLM as a Judge Files\test_results_df')

In [33]:
df_word_counts = df['text'].str.findall(r'\w+').str.len().sum()
df_word_counts += df['prompt'].str.findall(r'\w+').str.len().sum()

prompt_words = len(SYSTEM_PROMPT.split())

print(f'Word counts in dataframe: {df_word_counts}')
print(f'Word counts in prompt: {prompt_words} x 8400 = {prompt_words*8400}')

print(f'est. total tokens: {(df_word_counts + prompt_words*8400) / 3 * 4}')


Word counts in dataframe: 1833187
Word counts in prompt: 758 x 8400 = 6367200
est. total tokens: 10933849.333333334


#### Preparing batch

In [9]:
tasks = []

for index, row in df.iterrows():
    
    user_message = f"""
            Query:
            {row['prompt']}

            Result Abstract:
            {row['text']}

            Please evaluate the relevance according to the instructions and provide ONLY the JSON output.
            """
    
    task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4o-mini",
            "temperature": 0.1,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_message
                }
            ],
        }
    }
    
    tasks.append(task)

print(tasks[0])

{'custom_id': 'task-0', 'method': 'POST', 'url': '/v1/chat/completions', 'body': {'model': 'gpt-4o-mini', 'temperature': 0.1, 'response_format': {'type': 'json_object'}, 'messages': [{'role': 'system', 'content': '\nYou are tasked with evaluating the relevance of a conference program abstract (Result) to a specific search query (Query). Your evaluation must consider two distinct criteria: Application Relevance and Methodology Relevance.\n\nInput:\nYou will be given a Query and a Result.\n\nThe Query can be in various formats:\n- A set of keywords (e.g., "supply chain optimization")\n- A question (e.g., "How can simulation improve hospital workflows?")\n- A full conference abstract (where the goal is to find similar abstracts)\n\nThe Result will be a conference abstract.\n\nOutput:\nReturn a JSON object in the following format:\n{\n"Application": <application_score>,\n"Methodology": <methodology_score>\n}\n\nScoring Criteria:\nUse the following scale (1-3) to assign scores based on the 

In [ ]:
#splitting tasks into 12 parts (small enough batch sizes to be processed)

import math

def split_list_into_12_parts(data):
  """
  Splits a 1D list into 12 parts as evenly as possible.

  Args:
    data: The input 1D list.

  Returns:
    A list containing 12 sublists, or fewer if the original list has
    fewer than 12 elements (in which case each element becomes a part).
    Returns an empty list if the input list is empty.
  """
  if not data:
    return []

  list_len = len(data)
  num_parts = 12

  # If the list has fewer items than the desired number of parts,
  # return each item as a separate part.
  if list_len < num_parts:
      print(f"Warning: List length ({list_len}) is less than 12. Returning each element as a part.")
      return [[item] for item in data]

  base_part_size = list_len // num_parts
  remainder = list_len % num_parts

  parts = []
  start_index = 0
  for i in range(num_parts):
    # Determine the size of the current part
    part_size = base_part_size + (1 if i < remainder else 0)

    # Slice the list to get the current part
    end_index = start_index + part_size
    parts.append(data[start_index:end_index])

    # Update the start index for the next part
    start_index = end_index

  return parts

tasks_parts = split_list_into_12_parts(tasks)

print(len(tasks))

print(len(tasks_parts))

sum = 0
for p in tasks_parts:
   print(len(p))
   sum += len(p)

print(sum)


8400
12
700
700
700
700
700
700
700
700
700
700
700
700
8400


#### Initializing batches locally

In [ ]:
client = OpenAI(api_key=openai_key)

batch_id_lst = []

for i in range(0, len(tasks_parts)):
    tasks_batch = tasks_parts[i]
    file_name = f"./LLM as a Judge Files/testing_batches/num_{i}.jsonl"

    with open(file_name, 'w') as file:
        for obj in tasks_batch:
            file.write(json.dumps(obj) + '\n')

    batch_file = client.files.create(
        file=open(file_name, "rb"),
        purpose="batch"
        )
    
    batch_id_lst.append(batch_file.id)

print(batch_id_lst)

['file-AikZgkzbLfi6coawNUz6xK', 'file-Ne8HZa98HznjnvxWMEhChU', 'file-AKB46FwQQeXBfJfaoSwH8o', 'file-4X3KgjSF64GC3J6dt85Jaz', 'file-JArE8ZG7c8nZwEqXchrbwY', 'file-P2yE11BAzKocUZ5xifecYS', 'file-SZkmr17LTvgFEaz8JX2RPb', 'file-TwdofUtzpykxRzs9KYmZMJ', 'file-3U3rv9V3G8T3cZJDsmzP4C', 'file-9MTjV7f3JBfNtpdEyrsYPP', 'file-QmJPwAtz1D2CAfXsCrmdYx', 'file-Sj1uV9g4wNgWAf3MsTnFgt']


#### Sending batches to OpenAI.
Note that in the below code the batch number is hard coded into the file_id variable. This is because the batches would sometimes error out on the OpenAI side, and rerunning it would resolve this.

In [ ]:
import time
file_id = batch_id_lst[4]
check = True

while check:
    batch = client.batches.list(limit=100)
    in_progress_batches = [batch for batch in batch.data if batch.status == "in_progress"]
    if not in_progress_batches:
        batch_job = client.batches.create(
            input_file_id=file_id,
            endpoint="/v1/chat/completions",
            completion_window="24h"
            )
        check = False
        print(f"New batch queued: Number ")
    else:
        print(f"Current batch number still in progress")

New batch queued: Number 


#### Data Analysis

In [ ]:
batches = client.batches.list(limit=100)

# Extract only completed batch IDs
completed_batch_ids = [batch.output_file_id for batch in batches.data if batch.status == "completed"]
print(completed_batch_ids)
# filtered_batches = [item for i, item in enumerate(batch_id_lst) if i not in batches_left]

for i in range(0,len(completed_batch_ids)):
    file_response = client.files.content(completed_batch_ids[i]).content
    file_name = f"./LLM as a Judge Files/completed_batches/num_{i}.jsonl"

    with open(file_name, 'wb') as file:
        file.write(file_response)


['file-M1iX3HW1UHeKoPkYWojbfJ', 'file-Tw97dWFd3KgGRK6uqM9QYt', 'file-6kLmVCscDS4q3tjftj8gRT', 'file-7bbBnq4xtJAU8VtVQNey5u', 'file-M79b2sRndjVXHSksMjnf2p', 'file-K9KBaezNBEzAeLfhXuFLS2', 'file-A37tjX2fjUJpMUTFbABoih', 'file-8tgvw3GatxRXX4RXT7Xszu', 'file-14mmX4B1cf45B7EK9RKBq2', 'file-FbdpywsDMo5SzV3iFLwR7u', 'file-9TA11Mu8S2XMLUkhhqhVN4', 'file-YGDk3194Cr2n1nfjAauckv']


#### Creating new columns in dataframe representing application and methodology scores

Note that this section can be run modularly (e.g. you don't have to run the code above - the files output from above are already saved in the folder)

In [ ]:
score_df = pd.DataFrame()
for i in range(0,12):
    print(i)

    result_file_name = f"./LLM as a Judge Files/completed_batches/num_{i}.jsonl"
    results = []
    with open(result_file_name, 'r') as file:
        for line in file:
            # Parsing the JSON string into a dict and appending to the list of results
            json_object = json.loads(line.strip())
            results.append(json_object)

    for res in results:
        task_id = res['custom_id']
        # Getting index from task id
        index = int(task_id.split('-')[-1])
        scores = res['response']['body']['choices'][0]['message']['content']
        scores_dict = json.loads(scores)

        # Extract the scores using dictionary keys (use .get() for safety)
        application_score = scores_dict.get("Application")
        methodology_score = scores_dict.get("Methodology")

        temp_df = pd.DataFrame({'index': [index],
                                'application': [application_score],
                                'methodology': [methodology_score]})

        score_df = pd.concat([score_df, temp_df])

df_1 = pd.merge(df,score_df, right_on='index', left_index=True)
display(df_1)


0
1
2
3
4
5
6
7
8
9
10
11


,text,prompt,promptType,time,model_name,dataset,index,application,methodology
0,Quality Assurance Through Layered Process Audi...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,0,1,None
0,Enhancing supply chain resilience: evaluating ...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,1,3,3
0,How to succeed large organizational OPEX trans...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,2,1,None
0,Explore Your Career Choices. This presentatio...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,3,1,None
0,Industrial Engineers to the Rescue! Solving Yo...,supply chain resilience strategies,AI gen - short,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,4,2,2
...,...,...,...,...,...,...,...,...,...
0,Data-Driven Performance Guarantees for Classic...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8395,2,3
0,Distributionally Robust Optimization Through t...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8396,3,3
0,Optimization with Highly Adversarial Gradient ...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8397,1,2
0,High probability bounds for stochastic non-con...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8398,2,3


#### Getting summary data

In [23]:
cols_to_process = ['application', 'methodology']

# Replace string "None" with NaN
for col in cols_to_process:
    # Ensure the column exists before attempting replacement
    if col in df_1.columns:
        # Replace string "None" or None object with numpy NaN
        # Use a dictionary for multiple replacements including the None object
        df_1[col] = df_1[col].replace({'None': np.nan, None: np.nan})
        # Convert to numeric, coercing errors to NaN
        df_1[col] = pd.to_numeric(df_1[col], errors='coerce')

# Calculate 'average' column (mean of 'application' and 'methodology')
# The .mean(axis=1) will handle NaNs correctly
if 'application' in df_1.columns and 'methodology' in df_1.columns:
    df_1['average'] = df_1[['application', 'methodology']].mean(axis=1, skipna=True) # Explicitly skipna
else:
    print("Warning: 'application' or 'methodology' column not found. Cannot calculate 'average'.")
    df_1['average'] = np.nan

# Convert 'application' and 'methodology' to nullable integer type after calculations
# Using 'Int64' (capital I) allows for NaNs in integer columns
for col in cols_to_process:
    if col in df_1.columns:
        # Ensure the column is float first (pd.to_numeric already does this)
        # Convert to nullable integer
        # Note: mean calculation might result in floats, Int64 conversion truncates.
        # If you need floats, use float64 or remove this conversion.
        # For this example, we keep it as requested, but be aware of potential truncation.
         try:
            # Attempt conversion, handle potential type errors if all are NaN after coercion
             df_1[col] = df_1[col].astype('Int64')
         except TypeError:
             print(f"Warning: Could not convert column '{col}' to Int64. It might contain only NaNs or incompatible types.")


print("\nDataFrame after Step 1 processing:")
display(df_1)
print("-" * 30)


# Step 1.5: Consolidate 'promptType' categories
if 'promptType' in df_1.columns:
    print("\nConsolidating 'promptType' categories...")
    replace_dict = {
        'AI gen - short': 'AI gen',
        'AI gen - long': 'AI gen'
    }
    df_1['promptType'] = df_1['promptType'].replace(replace_dict)
    print("DataFrame after consolidating 'promptType':")
    display(df_1[['model_name', 'dataset', 'promptType', 'average']]) # Display relevant cols
    print("-" * 30)
else:
    print("Warning: 'promptType' column not found. Cannot consolidate categories.")


# Step 2: Group-wise calculations and pivoting

# Define grouping columns
grouping_cols = ['model_name', 'dataset']

# Check if grouping columns exist
if not all(col in df_1.columns for col in grouping_cols):
    print(f"Error: One or more grouping columns ({grouping_cols}) not found in the DataFrame. Stopping.")
    # Optionally: display(df_1) or raise ValueError("Missing grouping columns")
else:
    # Define columns for which to calculate the mean
    # Ensure 'average' exists before adding it to mean_cols
    mean_cols = ['time'] # Start with columns guaranteed to be numeric if they exist
    if 'average' in df_1.columns:
        mean_cols.append('average')
    # Add methodology and application only if they were successfully converted/exist
    if 'methodology' in df_1.columns and pd.api.types.is_numeric_dtype(df_1['methodology']):
         mean_cols.append('methodology')
    if 'application' in df_1.columns and pd.api.types.is_numeric_dtype(df_1['application']):
         mean_cols.append('application')

    # Ensure mean columns exist, filter list if some are missing
    mean_cols = [col for col in mean_cols if col in df_1.columns]

    grouped_means = pd.DataFrame() # Initialize an empty DF

    if not mean_cols:
        print("Warning: None of the specified numeric columns for mean calculation found or suitable.")
        # Create empty grouped_means with grouping cols if needed for merge structure
        if df_1.empty:
             grouped_means = pd.DataFrame(columns=grouping_cols)
        else:
             grouped_means = df_1[grouping_cols].drop_duplicates().reset_index(drop=True)

    else:
        # Calculate group-wise means for specified columns
        print(f"\nCalculating group means for columns: {mean_cols}")
        grouped_means = df_1.groupby(grouping_cols, observed=False)[mean_cols].mean().reset_index() # Use observed=False for potential categoricals

    # --- Pivoting Logic ---
    # Initialize df_result with grouped_means
    df_result = grouped_means

    # Check if 'promptType' and 'average' columns exist for pivoting
    if 'promptType' not in df_1.columns:
        print("Warning: 'promptType' column not found. Cannot create pivoted columns.")
    elif 'average' not in df_1.columns or df_1['average'].isnull().all():
        print("Warning: 'average' column not found or contains only NaNs. Cannot create pivoted columns based on 'average'.")
    else:
        print("\nCalculating pivot table based on consolidated 'promptType' and 'average'...")
        # Calculate mean of 'average' grouped by model, dataset, and CONSOLIDATED promptType
        # Ensure grouping_cols + ['promptType'] are present
        pivot_group_cols = grouping_cols + ['promptType']
        if not all(col in df_1.columns for col in pivot_group_cols):
             print(f"Warning: Missing columns required for pivoting {pivot_group_cols}. Skipping pivot.")
        else:
             # Use fillna(0) temporarily if you want 0 instead of NaN where a promptType doesn't exist for a group
             pivot_data = df_1.groupby(pivot_group_cols, observed=False)['average'].mean().reset_index() # Use observed=False for potential categoricals

             # Pivot the table to get CONSOLIDATED promptType categories as columns
             try:
                pivot_table = pivot_data.pivot_table(index=grouping_cols,
                                                    columns='promptType',
                                                    values='average').reset_index()

                # Rename the pivoted columns for clarity (optional, but good practice)
                pivot_table.columns.name = None # Remove the column index name 'promptType'
                # Add prefix to avoid potential name collisions with original columns if needed
                pivot_table = pivot_table.rename(columns={col: f'avg_prompt_{col}' for col in pivot_table.columns if col not in grouping_cols})

                # Merge the grouped means with the pivoted table
                # Check if grouped_means is not empty before merging
                if not grouped_means.empty:
                     print("\nMerging grouped means with pivot table...")
                     df_result = pd.merge(grouped_means, pivot_table, on=grouping_cols, how='left')
                else:
                     print("\nGrouped means table is empty, using only pivot table results.")
                     df_result = pivot_table # If no means were calculated, result is just the pivot

             except Exception as e:
                 print(f"Error during pivoting or merging: {e}")
                 print("Using intermediate grouped_means as result.")
                 df_result = grouped_means # Fallback to grouped_means if pivot fails

    # Display the final processed DataFrame
    print("\n--- Final Processed DataFrame ---")
    if 'df_result' in locals() and not df_result.empty:
        display(df_result)
        # Display DataFrame info (optional)
        print("\nFinal DataFrame Info:")
        df_result.info()
    else:
        print("Result DataFrame is empty or could not be generated.")


DataFrame after Step 1 processing:


,text,prompt,promptType,time,model_name,dataset,index,application,methodology,average
0,Quality Assurance Through Layered Process Audi...,supply chain resilience strategies,AI gen,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,0,1,<NA>,1.0
0,Enhancing supply chain resilience: evaluating ...,supply chain resilience strategies,AI gen,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,1,3,3,3.0
0,How to succeed large organizational OPEX trans...,supply chain resilience strategies,AI gen,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,2,1,<NA>,1.0
0,Explore Your Career Choices. This presentatio...,supply chain resilience strategies,AI gen,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,3,1,<NA>,1.0
0,Industrial Engineers to the Rescue! Solving Yo...,supply chain resilience strategies,AI gen,0.090018,snowflake-arctic-embed-m-v1.5,IISE 2024,4,2,2,2.0
...,...,...,...,...,...,...,...,...,...,...
0,Data-Driven Performance Guarantees for Classic...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8395,2,3,2.5
0,Distributionally Robust Optimization Through t...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8396,3,3,3.0
0,Optimization with Highly Adversarial Gradient ...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8397,1,2,1.5
0,High probability bounds for stochastic non-con...,The recent advent of data-driven and end-to-en...,From dataset,0.587680,all-MiniLM-L6-v2,INFORMS 2024,8398,2,3,2.5


------------------------------

Consolidating 'promptType' categories...
DataFrame after consolidating 'promptType':


,model_name,dataset,promptType,average
0,snowflake-arctic-embed-m-v1.5,IISE 2024,AI gen,1.0
0,snowflake-arctic-embed-m-v1.5,IISE 2024,AI gen,3.0
0,snowflake-arctic-embed-m-v1.5,IISE 2024,AI gen,1.0
0,snowflake-arctic-embed-m-v1.5,IISE 2024,AI gen,1.0
0,snowflake-arctic-embed-m-v1.5,IISE 2024,AI gen,2.0
...,...,...,...,...
0,all-MiniLM-L6-v2,INFORMS 2024,From dataset,2.5
0,all-MiniLM-L6-v2,INFORMS 2024,From dataset,3.0
0,all-MiniLM-L6-v2,INFORMS 2024,From dataset,1.5
0,all-MiniLM-L6-v2,INFORMS 2024,From dataset,2.5


------------------------------

Calculating group means for columns: ['time', 'average', 'methodology', 'application']

Calculating pivot table based on consolidated 'promptType' and 'average'...

Merging grouped means with pivot table...

--- Final Processed DataFrame ---


,model_name,dataset,time,average,methodology,application,avg_prompt_AI gen,avg_prompt_From dataset
0,all-MiniLM-L12-v2,IISE 2024,0.169761,2.212857,2.610248,1.895714,2.361,1.8425
1,all-MiniLM-L12-v2,INFORMS 2024,0.314133,2.389286,2.634731,2.186782,2.566,1.9475
2,all-MiniLM-L6-v2,IISE 2024,0.117600,2.223571,2.618683,1.895714,2.38,1.8325
3,all-MiniLM-L6-v2,INFORMS 2024,0.247066,2.414286,2.64,2.210678,2.599,1.9525
4,bge-small-en-v1.5,IISE 2024,0.075051,2.352143,2.684288,2.038571,2.513,1.95
5,bge-small-en-v1.5,INFORMS 2024,0.074087,2.590714,2.813314,2.373563,2.758,2.1725
6,granite-embedding-125m-english,IISE 2024,0.093041,2.365714,2.702541,2.061429,2.531,1.9525
7,granite-embedding-125m-english,INFORMS 2024,0.140829,2.571429,2.783505,2.375,2.72,2.2
8,snowflake-arctic-embed-m-v1.5,IISE 2024,0.085169,1.678571,2.365967,1.479885,1.595,1.8875
9,snowflake-arctic-embed-m-v1.5,INFORMS 2024,0.120825,1.553009,2.175926,1.371302,1.310241,2.1575



Final DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model_name               12 non-null     object 
 1   dataset                  12 non-null     object 
 2   time                     12 non-null     float64
 3   average                  12 non-null     Float64
 4   methodology              12 non-null     Float64
 5   application              12 non-null     Float64
 6   avg_prompt_AI gen        12 non-null     Float64
 7   avg_prompt_From dataset  12 non-null     Float64
dtypes: Float64(5), float64(1), object(2)
memory usage: 960.0+ bytes


#### Creating LaTeX table

In [14]:
grouped_df = df_result.groupby('model_name', as_index=False)

# Calculate the mean of 'average' and 'time' for each group
aggregated_results = grouped_df[['average', 'time']].mean()

aggregated_results = aggregated_results.rename(columns={
    'average': 'Mean Relevance Score',
    'time': 'Mean Time to Retrieve Results',
    'model_name': 'Model Name'
})

# Display the results
print("Aggregated Results DataFrame:")
display(aggregated_results)
print("\n" + "="*30 + "\n") # Separator

# --- LaTeX Conversion ---
# Convert the aggregated DataFrame to a LaTeX table string
# You might need to install the 'tabulate' library: pip install tabulate
# The .to_latex() method generates the LaTeX code for the table.
latex_table = aggregated_results.to_latex(float_format="%.4f") # Format floats to 4 decimal places

# Display the LaTeX table string
print("LaTeX Table Output:")
print(latex_table)

Aggregated Results DataFrame:


,Model Name,Mean Relevance Score,Mean Time to Retrieve Results
0,all-MiniLM-L12-v2,2.301071,0.241947
1,all-MiniLM-L6-v2,2.318929,0.182333
2,bge-small-en-v1.5,2.471429,0.074569
3,granite-embedding-125m-english,2.468571,0.116935
4,snowflake-arctic-embed-m-v1.5,1.615790,0.102997
5,snowflake-arctic-embed-s,1.995374,0.061959




LaTeX Table Output:
\begin{tabular}{llrr}
\toprule
 & Model Name & Mean Relevance Score & Mean Time to Retrieve Results \\
\midrule
0 & all-MiniLM-L12-v2 & 2.3011 & 0.2419 \\
1 & all-MiniLM-L6-v2 & 2.3189 & 0.1823 \\
2 & bge-small-en-v1.5 & 2.4714 & 0.0746 \\
3 & granite-embedding-125m-english & 2.4686 & 0.1169 \\
4 & snowflake-arctic-embed-m-v1.5 & 1.6158 & 0.1030 \\
5 & snowflake-arctic-embed-s & 1.9954 & 0.0620 \\
\bottomrule
\end{tabular}



In [24]:
cols_to_round = ['time', 'average', 'methodology', 'application', 'avg_prompt_AI gen', 'avg_prompt_From dataset']
df_result[cols_to_round] = df_result[cols_to_round].round(3)

# 3. Rename columns
rename_map = {
    'model_name': 'Model Name',
    'dataset': 'Dataset',
    'time': 'Mean Time to Retrieve Results',
    'average': 'Mean Overall Relevance Score',
    'methodology': 'Mean Methodology Relevance Score',
    'application': 'Mean Application Relevance Score',
    'avg_prompt_AI gen': 'Mean AI Generated Score',
    'avg_prompt_From dataset': 'Mean Score - Abstract from Dataset'
}
df_result = df_result.rename(columns=rename_map)

# 4. Prepare for LaTeX Export with Multi-level Header

# Define new column names (already done via rename_map values)
new_columns = list(rename_map.values())

# Define the multi-level structure
# First level: Empty string for columns before 'Prompt Category', then 'Prompt Category'
# Second level: The actual column names
header_level1 = [''] * 6 + ['Prompt Category'] * 2
header_level2 = new_columns

# Create the MultiIndex
df_result.columns = pd.MultiIndex.from_arrays([header_level1, header_level2])

# 5. Export to LaTeX
# Determine the number of columns for alignment string
num_columns = len(df_result.columns)
# Create the alignment string (center alignment 'c' for all columns)
alignment = 'c' * num_columns

# Generate LaTeX string
# multicolumn_format='c' ensures the 'Prompt Category' header itself is centered
latex_string = df_result.to_latex(index=False,
                           escape=False, # Generally good practice, might not be strictly needed here
                           multicolumn_format='c',
                           float_format="%.3f") # Center the spanning header ('Prompt Category')

# Print the LaTeX code
print(latex_string)

\begin{tabular}{llrrrrrr}
\toprule
\multicolumn{6}{c}{} & \multicolumn{2}{c}{Prompt Category} \\
Model Name & Dataset & Mean Time to Retrieve Results & Mean Overall Relevance Score & Mean Methodology Relevance Score & Mean Application Relevance Score & Mean AI Generated Score & Mean Score - Abstract from Dataset \\
\midrule
all-MiniLM-L12-v2 & IISE 2024 & 0.170 & 2.213 & 2.610 & 1.896 & 2.361 & 1.842 \\
all-MiniLM-L12-v2 & INFORMS 2024 & 0.314 & 2.389 & 2.635 & 2.187 & 2.566 & 1.948 \\
all-MiniLM-L6-v2 & IISE 2024 & 0.118 & 2.224 & 2.619 & 1.896 & 2.380 & 1.832 \\
all-MiniLM-L6-v2 & INFORMS 2024 & 0.247 & 2.414 & 2.640 & 2.211 & 2.599 & 1.952 \\
bge-small-en-v1.5 & IISE 2024 & 0.075 & 2.352 & 2.684 & 2.039 & 2.513 & 1.950 \\
bge-small-en-v1.5 & INFORMS 2024 & 0.074 & 2.591 & 2.813 & 2.374 & 2.758 & 2.172 \\
granite-embedding-125m-english & IISE 2024 & 0.093 & 2.366 & 2.703 & 2.061 & 2.531 & 1.952 \\
granite-embedding-125m-english & INFORMS 2024 & 0.141 & 2.571 & 2.784 & 2.375 & 2.720 &